In [3]:
!pip install dagster dagit pandas requests matplotlib

In [13]:
from dagster import job, op
import pandas as pd
import requests
from datetime import datetime
import os

# -----------------------------
# 1. Fetch Bitcoin Data (Simulated for 7 Days)
# -----------------------------

@op
def fetch_price_data():
    df = pd.read_csv("bitcoin_7day_data.csv")  # Use simulated data file
    return df

# -----------------------------
# 2. Transform Data
# -----------------------------

@op
def transform_data(df: pd.DataFrame):
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df.set_index('timestamp', inplace=True)
    df['moving_average'] = df['price'].rolling(window=12).mean()  # 12-hour MA
    return df

# -----------------------------
# 3. Store Data
# -----------------------------

@op
def store_data(df: pd.DataFrame):
    output_path = "processed_bitcoin_data.csv"
    df.to_csv(output_path)
    return output_path

# -----------------------------
# 4. Define the Dagster Job
# -----------------------------

@job
def bitcoin_price_job():
    store_data(transform_data(fetch_price_data()))

In [11]:
import matplotlib.pyplot as plt

df = pd.read_csv("processed_bitcoin_data.csv", parse_dates=['timestamp'], index_col='timestamp')
plt.figure(figsize=(14,6))
plt.plot(df['price'], label='Price')
plt.plot(df['moving_average'], label='12h Moving Average', linestyle='--')
plt.title("Bitcoin Price Over 7 Days with Moving Average")
plt.xlabel("Time")
plt.ylabel("Price (USD)")
plt.legend()
plt.grid(True)
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'processed_bitcoin_data.csv'